# Setup (Google Colab)

Run `data_and_training.ipynb` first so `models/` exists on Drive. Enable **GPU** runtime if available.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/DeepL-Final')
DATA_DIR = PROJECT_ROOT / 'Data' / 'caption_data'
IMAGES_DIR = DATA_DIR / 'Images'
CAPTIONS_FILE = DATA_DIR / 'captions.txt'
MODEL_DIR = PROJECT_ROOT / 'models'

assert (MODEL_DIR / 'best_model.pt').exists(), 'Train the model in data_and_training.ipynb first.'

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim

        resnet = models.resnet18(weights=None)
        modules = list(resnet.children())[:-1]
        self.encoder = nn.Sequential(*modules)
        self.feature_dim = 512

        self.init_h = nn.Linear(self.feature_dim, hidden_dim)
        self.init_c = nn.Linear(self.feature_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def encode_image(self, images):
        features = self.encoder(images)
        features = features.view(features.size(0), -1)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        return h0, c0

    def forward(self, images, captions):
        h0, c0 = self.encode_image(images)
        embeddings = self.dropout(self.embedding(captions))
        outputs, _ = self.lstm(embeddings, (h0, c0))
        logits = self.fc(self.dropout(outputs))
        return logits

In [ ]:
class TrainedCaptionModel:
    def __init__(self, model_dir=MODEL_DIR):
        self.artifacts = load_model_artifacts(model_dir)


def load_model_artifacts(model_dir=MODEL_DIR):
    with open(model_dir / 'config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    with open(model_dir / 'vocabulary.json', 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)

    word2idx = vocab_data['word2idx']
    idx2word = {int(k): v for k, v in vocab_data['idx2word'].items()}

    net = ImageCaptionModel(
        vocab_size=config['vocab_size'],
        embed_dim=config['embed_dim'],
        hidden_dim=config['hidden_dim'],
        pad_idx=config['pad_idx'],
    )
    state_dict = torch.load(model_dir / 'caption_model.pt', map_location=DEVICE)
    net.load_state_dict(state_dict)
    net.to(DEVICE)
    net.eval()

    transform = transforms.Compose([
        transforms.Resize((config['image_size'], config['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=config['normalize_mean'],
            std=config['normalize_std'],
        ),
    ])

    return {
        'model': net,
        'word2idx': word2idx,
        'idx2word': idx2word,
        'config': config,
        'transform': transform,
    }


def generate_caption(image_path: str, model: any) -> str:
    if isinstance(model, (str, Path)):
        artifacts = load_model_artifacts(Path(model))
    elif isinstance(model, dict):
        artifacts = model
    elif hasattr(model, 'artifacts'):
        artifacts = model.artifacts
    else:
        raise ValueError('model must be TrainedCaptionModel, artifacts dict, or model directory path')

    net = artifacts['model']
    word2idx = artifacts['word2idx']
    idx2word = artifacts['idx2word']
    config = artifacts['config']
    transform = artifacts['transform']

    start_idx = word2idx[config['start_token']]
    end_idx = word2idx[config['end_token']]
    max_len = config['max_seq_len']

    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(DEVICE)

    net.eval()
    with torch.no_grad():
        h, c = net.encode_image(image_tensor)
        current = torch.tensor([[start_idx]], device=DEVICE)
        words = []

        for _ in range(max_len):
            embedding = net.embedding(current)
            output, (h, c) = net.lstm(embedding, (h, c))
            logits = net.fc(output.squeeze(1))
            next_idx = logits.argmax(dim=-1).item()
            if next_idx == end_idx:
                break
            words.append(idx2word[next_idx])
            current = torch.tensor([[next_idx]], device=DEVICE)

    return ' '.join(words)

# Demonstration

In [ ]:
model = TrainedCaptionModel()

with open(MODEL_DIR / 'data_splits.json', 'r', encoding='utf-8') as f:
    splits = json.load(f)

test_images = splits['test_images']
caption_df = pd.read_csv(CAPTIONS_FILE)
reference_map = caption_df.groupby('image')['caption'].apply(list).to_dict()

random.seed(splits['seed'])
demo_images = random.sample(test_images, k=min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, image_name in zip(axes, demo_images):
    image_path = IMAGES_DIR / image_name
    generated = generate_caption(str(image_path), model)
    references = reference_map[image_name][:2]

    ax.imshow(Image.open(image_path).convert('RGB'))
    ax.axis('off')
    ref_text = ' | '.join(references)
    ax.set_title(
        f'Generated: {generated}\nReference: {ref_text}',
        fontsize=8,
    )

plt.suptitle('Test set caption generation', fontsize=14)
plt.tight_layout()
plt.show()

# Analysis

In [ ]:
def caption_overlap_score(generated, references):
    generated_words = set(generated.lower().split())
    best = 0.0
    for reference in references:
        reference_words = set(reference.lower().replace('.', ' ').split())
        if not reference_words:
            continue
        overlap = len(generated_words & reference_words) / len(reference_words)
        best = max(best, overlap)
    return best


eval_images = random.sample(test_images, k=min(120, len(test_images)))
results = []

for image_name in eval_images:
    image_path = IMAGES_DIR / image_name
    generated = generate_caption(str(image_path), model)
    references = reference_map[image_name]
    score = caption_overlap_score(generated, references)
    results.append(
        {
            'image': image_name,
            'generated': generated,
            'references': references,
            'score': score,
        }
    )

results = sorted(results, key=lambda item: item['score'], reverse=True)
success_cases = results[:3]
failure_cases = results[-3:]

print('Successful examples')
for case in success_cases:
    print(f"Image: {case['image']}")
    print(f"Generated: {case['generated']}")
    print(f"Reference: {case['references'][0]}")
    print(f"Word overlap score: {case['score']:.2f}")
    print()

print('Failure examples')
for case in failure_cases:
    print(f"Image: {case['image']}")
    print(f"Generated: {case['generated']}")
    print(f"Reference: {case['references'][0]}")
    print(f"Word overlap score: {case['score']:.2f}")
    print()

In [ ]:
def show_case_grid(cases, title):
    fig, axes = plt.subplots(1, len(cases), figsize=(5 * len(cases), 5))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        image_path = IMAGES_DIR / case['image']
        ax.imshow(Image.open(image_path).convert('RGB'))
        ax.axis('off')
        ax.set_title(
            f"Generated: {case['generated']}\nReference: {case['references'][0]}",
            fontsize=9,
        )
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_case_grid(success_cases, 'Successful captioning cases')
show_case_grid(failure_cases, 'Failure captioning cases')